# 03 - 表达矩阵质量控制

对下载的表达矩阵进行质量控制，包括：
- 矩阵形状和基因标识
- 缺失值检查
- 表达值分布
- 判断原始计数 vs 标准化表达

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_dir = Path('..')
processed_dir = base_dir / 'data' / 'processed'

# 查看示例数据
example_file = base_dir / 'data' / 'examples' / 'example_expression.csv'
df = pd.read_csv(example_file, index_col=0)
print(f'示例矩阵: {df.shape} (样本 × 基因)')
print(f'基因标识: Gene Symbol')
print(f'数据类型: {df.dtypes.iloc[0]}')

## 1. 缺失值检查

In [ ]:
missing = df.isnull().sum().sum()
total = df.size
print(f'缺失值: {missing} / {total} ({100*missing/total:.2f}%)')

if missing > 0:
    missing_by_gene = df.isnull().sum().sort_values(ascending=False)
    print(f'\n缺失最多的基因:')
    print(missing_by_gene[missing_by_gene > 0].head(10))

## 2. 表达值分布

In [ ]:
values = df.values.flatten()
print(f'表达值统计:')
print(f'  均值: {values.mean():.2f}')
print(f'  中位数: {np.median(values):.2f}')
print(f'  最小值: {values.min():.2f}')
print(f'  最大值: {values.max():.2f}')
print(f'  标准差: {values.std():.2f}')
print(f'  含负值: {(values < 0).any()}')
print(f'  全为整数: {np.all(values == values.astype(int))}')

# 判断表达类型
has_negative = (values < 0).any()
all_integers = np.all(values == values.astype(int))
max_val = values.max()
if all_integers and not has_negative and max_val > 100:
    print('\n表达类型: 原始计数')
elif has_negative or max_val <= 20:
    print('\n表达类型: 标准化表达 (log2 或 z-score)')
else:
    print('\n表达类型: 标准化表达 (非对数)')

## 3. 每个样本的总表达量

In [ ]:
sample_totals = df.sum(axis=1)
print(f'每个样本总表达量:')
print(sample_totals.describe())

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.bar(range(len(sample_totals)), sample_totals.values)
ax.set_xlabel('Sample index')
ax.set_ylabel('Total expression')
ax.set_title('Total expression per sample')
plt.tight_layout()
plt.show()

## 4. 基因标识类型检测

In [ ]:
def detect_id_type(columns):
    sample = [str(x) for x in columns[:50]]
    affy = sum(1 for x in sample if '_at' in x or x.startswith('AFFX-'))
    entrez = sum(1 for x in sample if x.isdigit())
    ensembl = sum(1 for x in sample if x.startswith('ENSG') or x.startswith('ENSMUSG'))
    if affy > len(sample) * 0.3:
        return 'affymetrix_probe'
    elif entrez > len(sample) * 0.8:
        return 'entrez_id'
    elif ensembl > len(sample) * 0.3:
        return 'ensembl_id'
    return 'gene_symbol'

id_type = detect_id_type(df.columns)
print(f'基因标识类型: {id_type}')
print(f'前10个基因: {list(df.columns[:10])}')

if id_type != 'gene_symbol':
    print('\n需要运行 harmonize_gene_ids.py 统一转换为 Gene Symbol')